# LeetCode 441 — Arranging Coins bằng Python

**Link đề:** https://leetcode.com/problems/arranging-coins/
**Độ khó:** Easy · **Chủ đề:** Math, Binary Search

```python
class Solution:
    def arrangeCoins(self, n: int) -> int:
        ...
```

## Mục tiêu học
1. Đổi bài toán "xếp đồng xu thành cầu thang" thành bất đẳng thức **`k(k+1)/2 <= n`** (tổng 1 + 2 + … + k).
2. Thấy được ba mức tư duy: **mô phỏng** `O(√n)` → **tìm kiếm nhị phân trên đáp án** (binary search on answer) `O(log n)` → **công thức nghiệm** `O(1)`.
3. Nắm template binary search "tìm `k` lớn nhất thỏa điều kiện" (`lo`, `hi`, `mid`, cập nhật biên) và tránh vòng lặp vô hạn.
4. Hiểu rủi ro **sai số số thực** với `math.sqrt` và cách dùng `math.isqrt` (căn nguyên chính xác).
5. Trace, chứng minh, Time/Space, lỗi hay gặp, biến thể.

## 1. Đề bài

Bạn có `n` đồng xu và muốn xếp thành **cầu thang**: hàng thứ `i` có **đúng** `i` đồng xu. Hàng cuối có thể **chưa đủ**.

Trả về **số hàng đầy đủ** của cầu thang.

| `n` | Output | Hình |
|:---:|:---:|---|
| `5` | `2` | hàng 1: ●, hàng 2: ●●, hàng 3: ●● (thiếu 1) |
| `8` | `3` | ● / ●● / ●●● / ●● (thiếu 2) |
| `1` | `1` | ● |
| `6` | `3` | ● / ●● / ●●● vừa đủ |

**Constraints:** `1 <= n <= 2^31 - 1`.

Câu hỏi đặt ra:
- Xếp đủ `k` hàng tốn bao nhiêu xu?
- `n` lên tới ~2.1 tỷ — mô phỏng từng hàng có nhanh không?
- Có công thức trực tiếp không? Nếu có, dùng căn bậc hai số thực có an toàn không?

## 2. Nền tảng: tổng 1 + 2 + … + k

Để có `k` hàng đầy đủ cần `1 + 2 + … + k` xu. Công thức (Gauss):

```text
S(k) = 1 + 2 + ... + k = k(k + 1) / 2
```

Mẹo nhớ: viết tổng xuôi và ngược rồi cộng từng cặp, mỗi cặp bằng `k + 1`, có `k` cặp → `2S = k(k+1)`.

| k | 1 | 2 | 3 | 4 | 5 | 10 |
|---|---|---|---|---|---|---|
| S(k) | 1 | 3 | 6 | 10 | 15 | 55 |

**Đáp án** = số `k` **lớn nhất** sao cho `S(k) <= n`.

In [1]:
def S(k: int) -> int:
    return k * (k + 1) // 2

print([S(k) for k in range(1, 11)])
print("tính bằng vòng lặp S(10) =", sum(range(1, 11)), "| công thức =", S(10))

[1, 3, 6, 10, 15, 21, 28, 36, 45, 55]
tính bằng vòng lặp S(10) = 55 | công thức = 55


Chú ý `//` (chia nguyên): `k(k+1)` luôn **chẵn** (một trong hai số liên tiếp là chẵn), nên `k * (k + 1) // 2` là số nguyên chính xác. Dùng `/` sẽ ra số thực `55.0`.

### 2.1. Số nguyên Python không tràn

Trong C++/Java, `k * (k + 1)` với `k ≈ 65536` vượt kiểu `int` 32-bit. Python tự mở rộng số nguyên nên không tràn — nhưng khi phỏng vấn nên biết điểm này.

In [2]:
k = 65536
print(k * (k + 1), "> 2^31 - 1 =", 2**31 - 1, "?", k * (k + 1) > 2**31 - 1)
print(10 / 2, 10 // 2, type(10 / 2), type(10 // 2))

4295032832 > 2^31 - 1 = 2147483647 ? True
5.0 5 <class 'float'> <class 'int'>


## 3. Cách trực giác (baseline) — mô phỏng từng hàng

Xếp hàng 1, 2, 3… cho tới khi không đủ xu cho hàng tiếp theo:

```python
row = 0
while n >= row + 1:   # còn đủ cho hàng kế tiếp?
    row += 1
    n -= row
return row
```

In [3]:
def arrange_coins_simulate(n: int) -> int:
    row = 0
    while n >= row + 1:
        row += 1
        n -= row
    return row

for n in [1, 5, 6, 8, 10]:
    print(n, "->", arrange_coins_simulate(n))

1 -> 1
5 -> 2
6 -> 3
8 -> 3
10 -> 4


### 3.1. Chi phí của mô phỏng

Số vòng = đáp án `k ≈ √(2n)`. Với `n = 2^31 - 1`, `k ≈ 65535` vòng — vẫn chạy được trên LeetCode, nhưng là `O(√n)`. Ta đo thử:

In [4]:
import time
n = 2**31 - 1
t0 = time.perf_counter()
k = arrange_coins_simulate(n)
t1 = time.perf_counter()
print("n =", n, "-> k =", k, f"| số vòng ≈ {k} | thời gian ≈ {(t1 - t0) * 1000:.1f} ms")

n = 2147483647 -> k = 65535 | số vòng ≈ 65535 | thời gian ≈ 4.1 ms


## 4. Nút thắt và insight

**Nút thắt:** mô phỏng thử **từng** `k` một. Nhưng điều kiện `S(k) <= n` có tính **đơn điệu**: `S(k)` tăng khi `k` tăng. Nên dãy đúng/sai trông như:

```text
k:        0  1  2  3  4  5  6 ...
S(k)<=8?  T  T  T  T  F  F  F ...
                   ^ đáp án = vị trí T cuối cùng
```

**Insight 1 — binary search trên đáp án:** với dãy kiểu `T T T F F F`, tìm điểm chuyển bằng cách **chia đôi** khoảng tìm kiếm mỗi bước → `O(log n)`.

**Insight 2 — giải bất phương trình:** `k² + k - 2n <= 0` ⟺ `k <= (√(8n + 1) - 1) / 2`. Đáp án = phần nguyên của vế phải → `O(1)`. Cần cẩn thận sai số số thực.

In [5]:
n = 8
print([(k, S(k), S(k) <= n) for k in range(0, 7)])

[(0, 0, True), (1, 1, True), (2, 3, True), (3, 6, True), (4, 10, False), (5, 15, False), (6, 21, False)]


## 5. Lời giải chính — binary search "k lớn nhất thỏa `S(k) <= n`"

**Đoạn 1 — khoảng tìm kiếm:** đáp án nằm trong `[0, n]` (vì `S(n) >= n`). Đặt `lo = 0`, `hi = n`.

**Đoạn 2 — vòng lặp** `while lo < hi`:
```python
mid = (lo + hi + 1) // 2       # làm tròn LÊN
if S(mid) <= n:
    lo = mid                   # mid thỏa -> đáp án >= mid
else:
    hi = mid - 1               # mid không thỏa -> đáp án < mid
```

**Đoạn 3 — kết luận:** khi `lo == hi`, đó là đáp án.

**Vì sao làm tròn lên?** Khi `hi = lo + 1`, nếu dùng `(lo + hi) // 2 = lo` và nhánh `lo = mid` xảy ra thì `lo` không đổi → **lặp vô hạn**. Làm tròn lên đảm bảo `mid > lo`, nên mỗi vòng khoảng `[lo, hi]` luôn **thu hẹp**.

In [6]:
def arrange_coins_binary(n: int) -> int:
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if mid * (mid + 1) // 2 <= n:
            lo = mid
        else:
            hi = mid - 1
    return lo

for n in [1, 5, 6, 8, 10, 2**31 - 1]:
    print(n, "->", arrange_coins_binary(n))

1 -> 1
5 -> 2
6 -> 3
8 -> 3
10 -> 4
2147483647 -> 65535


## 6. TRACE từng bước

Trace `n = 8` (đáp án 3):

| Vòng | lo | hi | mid | S(mid) | S(mid) <= 8? | Cập nhật |
|:---:|:---:|:---:|:---:|:---:|:---:|---|
| 1 | 0 | 8 | 4 | 10 | không | hi = 3 |
| 2 | 0 | 3 | 2 | 3 | có | lo = 2 |
| 3 | 2 | 3 | 3 | 6 | có | lo = 3 |
| — | 3 | 3 | | | | dừng, trả 3 |

Ở vòng 3, `lo = 2`, `hi = 3`: làm tròn lên cho `mid = 3` (nếu làm tròn xuống thì `mid = 2` → `lo = 2` mãi mãi).

In [7]:
def trace_binary(n: int) -> int:
    lo, hi = 0, n
    step = 1
    print(f"n = {n}")
    while lo < hi:
        mid = (lo + hi + 1) // 2
        s = mid * (mid + 1) // 2
        if s <= n:
            note = f"S({mid})={s} <= {n} -> lo = {mid}"
            lo = mid
        else:
            note = f"S({mid})={s} >  {n} -> hi = {mid - 1}"
            hi = mid - 1
        print(f"  vòng {step}: mid={mid:<6} {note:<34} | [lo, hi] = [{lo}, {hi}]")
        step += 1
    print("  kết quả:", lo)
    return lo

trace_binary(8)
print()
trace_binary(2**31 - 1)

n = 8
  vòng 1: mid=4      S(4)=10 >  8 -> hi = 3             | [lo, hi] = [0, 3]
  vòng 2: mid=2      S(2)=3 <= 8 -> lo = 2              | [lo, hi] = [2, 3]
  vòng 3: mid=3      S(3)=6 <= 8 -> lo = 3              | [lo, hi] = [3, 3]
  kết quả: 3

n = 2147483647
  vòng 1: mid=1073741824 S(1073741824)=576460752840294400 >  2147483647 -> hi = 1073741823 | [lo, hi] = [0, 1073741823]
  vòng 2: mid=536870912 S(536870912)=144115188344291328 >  2147483647 -> hi = 536870911 | [lo, hi] = [0, 536870911]
  vòng 3: mid=268435456 S(268435456)=36028797153181696 >  2147483647 -> hi = 268435455 | [lo, hi] = [0, 268435455]
  vòng 4: mid=134217728 S(134217728)=9007199321849856 >  2147483647 -> hi = 134217727 | [lo, hi] = [0, 134217727]
  vòng 5: mid=67108864 S(67108864)=2251799847239680 >  2147483647 -> hi = 67108863 | [lo, hi] = [0, 67108863]
  vòng 6: mid=33554432 S(33554432)=562949970198528 >  2147483647 -> hi = 33554431 | [lo, hi] = [0, 33554431]
  vòng 7: mid=16777216 S(16777216)=140737496743936 > 

65535

Với `n ≈ 2.1 tỷ` chỉ cần khoảng **31 vòng** (vì `2^31` ≈ n và mỗi vòng chia đôi), so với ~65 nghìn vòng mô phỏng.

## 7. Các lời giải khác

### 7.1. Cách 3 — công thức nghiệm với `math.isqrt` (chính xác tuyệt đối)

Giải `k² + k - 2n <= 0`. Nghiệm dương của `k² + k - 2n = 0` là `k* = (√(8n + 1) - 1) / 2`. Đáp án = `⌊k*⌋`.

`math.isqrt(m)` trả **phần nguyên của căn bậc hai** của số nguyên `m`, tính bằng số nguyên → **không có sai số**. Và ta có `⌊(√(8n+1) - 1)/2⌋ = (isqrt(8n + 1) - 1) // 2`.

In [8]:
import math

print("isqrt(17) =", math.isqrt(17), "| sqrt(17) =", math.sqrt(17))
print("isqrt(16) =", math.isqrt(16))

def arrange_coins_isqrt(n: int) -> int:
    return (math.isqrt(8 * n + 1) - 1) // 2

for n in [1, 5, 6, 8, 10, 2**31 - 1]:
    print(n, "->", arrange_coins_isqrt(n))

isqrt(17) = 4 | sqrt(17) = 4.123105625617661
isqrt(16) = 4
1 -> 1
5 -> 2
6 -> 3
8 -> 3
10 -> 4
2147483647 -> 65535


### 7.2. Cách 4 — công thức với `math.sqrt` (số thực) + hiệu chỉnh

Nhiều lời giải viết `int((math.sqrt(8 * n + 1) - 1) / 2)`. Với `n <= 2^31 - 1` thì `8n + 1 < 2^34`, nằm trong vùng `float` (53 bit) biểu diễn chính xác, và `sqrt` của IEEE-754 được làm tròn đúng — nên thực tế cho kết quả đúng trong constraints. Nhưng với `n` **rất lớn**, nhất là `n` nằm sát ranh giới `S(k) - 1`, số thực mất chính xác và có thể lệch 1. Ví dụ `n = S(10^9) - 1 = 500000000499999999`: đáp án đúng là `10^9 - 1`.

Cách an toàn: tính bằng float rồi **hiệu chỉnh** bằng số nguyên: tăng/giảm `k` cho tới khi `S(k) <= n < S(k+1)`.

In [9]:
def arrange_coins_float(n: int) -> int:
    return int((math.sqrt(8 * n + 1) - 1) / 2)

def arrange_coins_float_fixed(n: int) -> int:
    k = int((math.sqrt(8 * n + 1) - 1) / 2)
    while k * (k + 1) // 2 > n:            # float đoán dư
        k -= 1
    while (k + 1) * (k + 2) // 2 <= n:     # float đoán thiếu
        k += 1
    return k

big = S(10 ** 9) - 1
print("n =", big)
print("float      :", arrange_coins_float(big))
print("float+fix  :", arrange_coins_float_fixed(big))
print("isqrt      :", arrange_coins_isqrt(big))
k = arrange_coins_float(big)
print("float có thỏa S(k) <= n < S(k+1)?", S(k) <= big < S(k + 1))

n = 500000000499999999
float      : 1000000000
float+fix  : 999999999
isqrt      : 999999999
float có thỏa S(k) <= n < S(k+1)? False


Kết quả cho thấy: với `n = S(10^9) - 1`, công thức float **sai** (cho `10^9`, dư 1 so với `isqrt`), còn bản hiệu chỉnh và `isqrt` đúng. Bài học: khi làm việc với số nguyên lớn, **tránh** số thực hoặc luôn kiểm tra lại bằng số nguyên.

## 8. Code nộp LeetCode

Bản chính: binary search (kỹ năng dùng lại được cho nhiều bài). Bản phụ: công thức `isqrt` `O(1)`.

In [10]:
class Solution:
    def arrangeCoins(self, n: int) -> int:
        lo, hi = 0, n
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if mid * (mid + 1) // 2 <= n:
                lo = mid
            else:
                hi = mid - 1
        return lo


sol = Solution()
print(sol.arrangeCoins(5), sol.arrangeCoins(8))   # 2 3

2 3


In [11]:
class SolutionMath:
    def arrangeCoins(self, n: int) -> int:
        return (math.isqrt(8 * n + 1) - 1) // 2

print(SolutionMath().arrangeCoins(5), SolutionMath().arrangeCoins(8))

2 3


## 9. Vì sao đúng? (invariant)

**Binary search.** Gọi `k*` là đáp án (lớn nhất với `S(k*) <= n`). **Bất biến:** `lo <= k* <= hi` và `S(lo) <= n`.
- Ban đầu `lo = 0` (`S(0) = 0 <= n`), `hi = n` (`S(n+1) > n` nên `k* <= n`) → đúng.
- Nếu `S(mid) <= n`: do `S` tăng, `k* >= mid` → gán `lo = mid` giữ bất biến.
- Nếu `S(mid) > n`: mọi `k >= mid` đều sai → `k* <= mid - 1` → gán `hi = mid - 1` giữ bất biến.
- `mid` thuộc `(lo, hi]` (làm tròn lên) nên khoảng thu hẹp ít nhất 1 mỗi vòng → vòng lặp dừng, khi đó `lo == hi == k*`. ∎

**Mô phỏng.** Sau vòng thứ `r`, đã xếp đủ `r` hàng và `n` còn lại `= n_gốc - S(r)`. Vòng tiếp chạy khi còn đủ `r + 1` xu ⟺ `S(r+1) <= n_gốc`. Dừng tại `r` lớn nhất thỏa → đúng `k*`.

**Công thức.** `k² + k - 2n <= 0` ⟺ `k <= k*_thực`; số nguyên lớn nhất là `⌊k*_thực⌋`. Với `m = 8n + 1`: `⌊(√m - 1)/2⌋ = ⌊(⌊√m⌋ - 1)/2⌋` (vì `(x - 1)/2` tăng và các ngưỡng nguyên của nó ứng với `√m` nguyên).

## 10. Độ phức tạp

Gọi `n` là số xu.

| Cách | Time | Space |
|---|:---:|:---:|
| Mô phỏng | `O(√n)` (~65 nghìn vòng với n tối đa) | `O(1)` |
| **Binary search** | `O(log n)` (~31 vòng) | `O(1)` |
| `isqrt` | `O(1)` với số 64-bit (thực chất `O(log n)` phép toán bit bên trong) | `O(1)` |
| `sqrt` float | `O(1)` — nhưng rủi ro sai số với `n` lớn | `O(1)` |

## 11. Bộ test

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<32} -> {actual}")

sol = Solution()
cases = [
    ("đề 1", 5, 2), ("đề 2", 8, 3), ("n = 1", 1, 1), ("n = 2", 2, 1), ("n = 3 vừa đủ", 3, 2),
    ("n = 6 vừa đủ", 6, 3), ("n = 9 thiếu 1", 9, 3), ("n = 10 vừa đủ", 10, 4),
    ("n max", 2**31 - 1, 65535), ("S(65535) vừa đủ", 65535 * 65536 // 2, 65535),
    ("S(65535) - 1", 65535 * 65536 // 2 - 1, 65534),
]
for name, n, exp in cases:
    check(name, sol.arrangeCoins(n), exp)

PASS  đề 1                             -> 2
PASS  đề 2                             -> 3
PASS  n = 1                            -> 1
PASS  n = 2                            -> 1
PASS  n = 3 vừa đủ                     -> 2
PASS  n = 6 vừa đủ                     -> 3
PASS  n = 9 thiếu 1                    -> 3
PASS  n = 10 vừa đủ                    -> 4
PASS  n max                            -> 65535
PASS  S(65535) vừa đủ                  -> 65535
PASS  S(65535) - 1                     -> 65534


In [13]:
import random
rng = random.Random(441)
funcs = [arrange_coins_binary, arrange_coins_isqrt, arrange_coins_float, arrange_coins_float_fixed,
         Solution().arrangeCoins, SolutionMath().arrangeCoins]
for n in range(1, 3000):
    exp = arrange_coins_simulate(n)
    for f in funcs:
        assert f(n) == exp, (f.__name__, n)
for _ in range(3000):
    n = rng.randint(1, 2**31 - 1)
    exp = arrange_coins_isqrt(n)
    assert S(exp) <= n < S(exp + 1)
    for f in funcs:
        assert f(n) == exp, (f.__name__, n)
# ranh giới S(k) và S(k) - 1 cho mọi k trong constraints (bẫy hay gặp của float)
for k in range(1, 65536):
    for n in (S(k) - 1, S(k)):
        if 1 <= n <= 2**31 - 1:
            assert arrange_coins_float(n) == arrange_coins_isqrt(n) == arrange_coins_binary(n)
print("n = 1..2999, 3000 n ngẫu nhiên, và mọi ranh giới S(k): tất cả khớp -> PASS")

n = 1..2999, 3000 n ngẫu nhiên, và mọi ranh giới S(k): tất cả khớp -> PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Trả về số hàng **đã bắt đầu** (kể cả hàng thiếu)
```python
while n > 0:
    row += 1; n -= row
return row
```
`n = 5` → 3 (sai, hàng 3 chưa đủ). Phải kiểm tra **đủ xu trước** khi xếp.

### Lỗi 2: Binary search làm tròn xuống với `lo = mid`
`mid = (lo + hi) // 2` + `lo = mid` → lặp vô hạn khi `hi = lo + 1` và `mid` thỏa.

In [14]:
def wrong_count_partial(n):
    row = 0
    while n > 0:
        row += 1
        n -= row
    return row

def wrong_binary_round_down(n, max_steps=50):
    lo, hi = 0, n
    steps = 0
    while lo < hi:
        mid = (lo + hi) // 2
        if mid * (mid + 1) // 2 <= n:
            lo = mid
        else:
            hi = mid - 1
        steps += 1
        if steps >= max_steps:                     # chốt an toàn để notebook không treo
            return f"kẹt ở lo={lo}, hi={hi} sau {steps} vòng"
    return lo

print("lỗi 1, n=5:", wrong_count_partial(5), "(đúng 2)")
print("lỗi 2, n=8:", wrong_binary_round_down(8))

lỗi 1, n=5: 3 (đúng 2)
lỗi 2, n=8: kẹt ở lo=2, hi=3 sau 50 vòng


### Lỗi 3: Dùng `/` thay vì `//` khi tính `S(k)`
Kết quả là float; với số lớn có thể mất chính xác, và so sánh float với int dễ gây lỗi khó thấy.

### Lỗi 4: Tin tuyệt đối vào `math.sqrt`
Đúng trong constraints của đề (như đã kiểm tra ở mục 11), nhưng sai với `n` lớn hơn (mục 7.2). Dùng `isqrt` hoặc hiệu chỉnh.

### Lỗi 5: Khoảng tìm kiếm sai — `hi` quá nhỏ
Đặt `hi = n // 2` trông "tối ưu" nhưng với `n = 1` đáp án là `1 > 0` → sai.

In [15]:
def wrong_small_hi(n):
    lo, hi = 0, n // 2
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if mid * (mid + 1) // 2 <= n:
            lo = mid
        else:
            hi = mid - 1
    return lo

print("lỗi 5, n=1:", wrong_small_hi(1), "(đúng 1) | n=2:", wrong_small_hi(2), "(đúng 1)")
print("lỗi 3:", 10**30 * (10**30 + 1) / 2 == 10**30 * (10**30 + 1) // 2)

lỗi 5, n=1: 0 (đúng 1) | n=2: 1 (đúng 1)
lỗi 3: False


## 13. Mẹo nhận dạng pattern

- "Tìm **k lớn nhất / nhỏ nhất** sao cho `f(k) <= n`" với `f` **đơn điệu** → **binary search trên đáp án**.
- Template "lớn nhất thỏa": `mid = (lo + hi + 1) // 2`, thỏa → `lo = mid`, không thỏa → `hi = mid - 1`.
- Template "nhỏ nhất thỏa": `mid = (lo + hi) // 2`, thỏa → `hi = mid`, không thỏa → `lo = mid + 1`.
- Có tổng `1 + 2 + … + k` → công thức `k(k+1)/2`; bất phương trình bậc hai → công thức nghiệm + `isqrt`.

Bài liên quan: 69 Sqrt(x), 367 Valid Perfect Square, 374 Guess Number Higher or Lower, 875 Koko Eating Bananas, 1011 Capacity To Ship Packages.

## 14. Biến thể

1. **LeetCode 69 — Sqrt(x):** `k` lớn nhất với `k² <= x` — cùng template.
2. **Cầu thang mỗi hàng `d` xu nhiều hơn hàng trước** (cấp số cộng): `S(k) = k·a + d·k(k-1)/2`.
3. **Số xu còn thiếu** để hoàn thành hàng tiếp theo: `S(k+1) - n`.

In [16]:
def my_sqrt(x: int) -> int:
    lo, hi = 0, x
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if mid * mid <= x:
            lo = mid
        else:
            hi = mid - 1
    return lo

def coins_missing(n: int) -> int:
    k = arrange_coins_isqrt(n)
    return S(k + 1) - n

print("sqrt:", [my_sqrt(x) for x in [0, 1, 8, 16, 2**31 - 1]], "| isqrt:", math.isqrt(2**31 - 1))
print("thiếu bao nhiêu xu:", [(n, coins_missing(n)) for n in [5, 6, 8]])

sqrt: [0, 1, 2, 4, 46340] | isqrt: 46340
thiếu bao nhiêu xu: [(5, 1), (6, 4), (8, 2)]


## 15. Bài tập (TODO)

**Bài 1 (tính tay):** Trace binary search cho `n = 20` (bảng `lo, hi, mid, S(mid)`).

**Bài 2 (đoán output):** `arrange_coins_simulate(15)`, `arrange_coins_simulate(14)`?

**Bài 3 (điền chỗ trống):** hoàn thiện binary search.

In [17]:
def exercise_arrange_coins(n: int) -> int:
    lo, hi = 0, n
    while lo < hi:
        # TODO 1: mid làm tròn lên
        # TODO 2: nếu S(mid) <= n -> lo = mid, ngược lại hi = mid - 1
        pass
    return lo

# assert exercise_arrange_coins(5) == 2
# assert exercise_arrange_coins(8) == 3

**Bài 4 (tự viết):** `exercise_my_sqrt(x)` — phần nguyên căn bậc hai của `x >= 0` bằng binary search, không dùng `math`.

**Bài 5 (biến thể):** `exercise_arith_staircase(n, a, d)` — hàng 1 có `a` xu, mỗi hàng sau nhiều hơn `d` xu (`a >= 1`, `d >= 0`). Trả số hàng đầy đủ.

**Bài 6 (mini challenge):** `exercise_min_coins_for_rows(k)` và `exercise_rows_needed(target_rows, n)`: cho `n` xu, cần **thêm** ít nhất bao nhiêu xu để có đủ `target_rows` hàng (0 nếu đã đủ)?

In [18]:
def exercise_my_sqrt(x: int) -> int:
    # TODO
    pass

def exercise_arith_staircase(n: int, a: int, d: int) -> int:
    # TODO
    pass

def exercise_rows_needed(target_rows: int, n: int) -> int:
    # TODO
    pass

# assert exercise_my_sqrt(8) == 2
# assert exercise_arith_staircase(10, 1, 1) == 4
# assert exercise_rows_needed(4, 8) == 2

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Bắt đầu `[0, 20]` | `mid = 10`, `S(10) = 55 > 20` | Đáp án là 5 vì `S(5) = 15 <= 20 < 21 = S(6)` |
| 2 | Tính `S(k)` | `S(5) = 15` | 15 vừa đủ, 14 thiếu |
| 3 | Theo mục 5 | `mid = (lo + hi + 1) // 2` | `if mid*(mid+1)//2 <= n: lo = mid else: hi = mid - 1` |
| 4 | Như mục 14 | Điều kiện `mid * mid <= x` | Khoảng `[0, x]` |
| 5 | Viết `S(k)` mới | `S(k) = k*a + d*k*(k-1)//2` | Binary search với `hi = n` |
| 6 | Cần `S(target_rows)` xu | So với `n` | `max(0, S(target_rows) - n)` |

## 17. Đáp án tham khảo

- **Bài 1:** `[0,20]` mid=10 S=55>20 → hi=9; `[0,9]` mid=5 S=15≤20 → lo=5; `[5,9]` mid=7 S=28>20 → hi=6; `[5,6]` mid=6 S=21>20 → hi=5 → **5**.
- **Bài 2:** `15 = S(5)` → **5**; `14` → **4**.

In [19]:
trace_binary(20)
print("Bài 2:", arrange_coins_simulate(15), arrange_coins_simulate(14))

n = 20
  vòng 1: mid=10     S(10)=55 >  20 -> hi = 9           | [lo, hi] = [0, 9]
  vòng 2: mid=5      S(5)=15 <= 20 -> lo = 5            | [lo, hi] = [5, 9]
  vòng 3: mid=7      S(7)=28 >  20 -> hi = 6            | [lo, hi] = [5, 6]
  vòng 4: mid=6      S(6)=21 >  20 -> hi = 5            | [lo, hi] = [5, 5]
  kết quả: 5
Bài 2: 5 4


In [20]:
def answer_arrange_coins(n: int) -> int:
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if mid * (mid + 1) // 2 <= n:
            lo = mid
        else:
            hi = mid - 1
    return lo

def answer_my_sqrt(x: int) -> int:
    lo, hi = 0, x
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if mid * mid <= x:
            lo = mid
        else:
            hi = mid - 1
    return lo

def answer_arith_staircase(n: int, a: int, d: int) -> int:
    def cost(k):
        return k * a + d * k * (k - 1) // 2
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if cost(mid) <= n:
            lo = mid
        else:
            hi = mid - 1
    return lo

def answer_rows_needed(target_rows: int, n: int) -> int:
    return max(0, S(target_rows) - n)

assert answer_arrange_coins(5) == 2 and answer_arrange_coins(8) == 3
assert all(answer_my_sqrt(x) == math.isqrt(x) for x in range(0, 5000))
assert answer_arith_staircase(10, 1, 1) == 4 and answer_arith_staircase(10, 2, 0) == 5
assert answer_arith_staircase(5, 1, 1) == arrange_coins_simulate(5)
assert answer_rows_needed(4, 8) == 2 and answer_rows_needed(3, 8) == 0
print("Đáp án bài 3–6 PASS")

Đáp án bài 3–6 PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def arrangeCoins(self, n: int) -> int:
        lo, hi = 0, n                       # đáp án trong [0, n]
        while lo < hi:
            mid = (lo + hi + 1) // 2        # tìm LỚN NHẤT -> làm tròn lên
            if mid * (mid + 1) // 2 <= n:   # S(mid) <= n
                lo = mid
            else:
                hi = mid - 1
        return lo
# O(1): return (math.isqrt(8 * n + 1) - 1) // 2
```

**Checklist:**
- [ ] Chỉ đếm hàng **đầy đủ**.
- [ ] `S(k) = k * (k + 1) // 2` (chia nguyên).
- [ ] Binary search "lớn nhất thỏa": `mid` làm tròn lên, `lo = mid` / `hi = mid - 1`.
- [ ] Khoảng `[0, n]` bao trọn đáp án (kể cả `n = 1`).
- [ ] Nếu dùng công thức: ưu tiên `math.isqrt`, tránh sai số float.
- [ ] Time `O(log n)` (hoặc `O(1)`), Space `O(1)`.